In [1]:
import numpy as np 
import pandas as pd 

def optimal_weights(sigma,mu):
    wgt = np.linalg.inv(sigma) @ mu 
    wgt = wgt / np.abs(wgt).sum()
    return wgt

def eqvol_weights(sigma):
    wgt = 1/np.sqrt(np.diag(sigma))
    wgt = wgt / np.abs(wgt).sum()
    return wgt

def sr_weights(sigma,mu):
    wgt = mu / np.diag(sigma) 
    wgt = wgt / np.abs(wgt).sum()
    return wgt

def gen_strat_returns():
    np.random.seed(5)
    
    corr = [[1, 0.3, 0],
            [0.3, 1, 0],
            [0,   0, 1]]
    
    corr = np.array(corr)
    
    vols = np.diag(np.array([0.1, 0.06, 0.02])) / np.sqrt(252)
    
    sigma = vols @ corr @ vols
    
    mu = np.array([0.1,0.12,0.04]) / 252
    
    dates = pd.date_range('20100101','20191231',freq='B')
    
    rets = np.random.multivariate_normal(mu, sigma, size = len(dates))
    rets = pd.DataFrame(rets,columns = ['A','B','C'], index = dates)
    return rets

In [2]:
# ...existing code...
def compute_mu_sigma(rets, annualize=True, periods=252):
    mu = rets.mean()
    sigma = rets.cov()
    if annualize:
        mu = mu * periods
        sigma = sigma * periods
    return mu, sigma

def add_stock_to_rets(rets, name, returns_series=None, prices=None):
    """
    returns_series : pd.Series of returns (index = dates)
    prices         : pd.Series of prices (will be converted to returns)
    """
    if returns_series is None and prices is None:
        raise ValueError("Provide returns_series or prices")
    if returns_series is None:
        returns_series = prices.pct_change().dropna()
    s = returns_series.copy().rename(name)
    s.index = pd.to_datetime(s.index)
    new_rets = pd.concat([rets, s], axis=1)
    new_rets = new_rets.dropna(how='all')    # remove rows with no data at all
    return new_rets

def recompute_weights_from_rets(rets, periods=252):
    mu, sigma = compute_mu_sigma(rets, annualize=True, periods=periods)
    sigma_mat = sigma.values
    mu_vec = mu.values
    w_opt = optimal_weights(sigma_mat, mu_vec)
    w_eq = eqvol_weights(sigma_mat)
    w_sr  = sr_weights(sigma_mat, mu_vec)
    df = pd.DataFrame({
        'opt': w_opt,
        'eqvol': w_eq,
        'sr': w_sr
    }, index=rets.columns)
    return df

# Example usage:
# 1) If you have a pd.Series `candidate_ret` of returns (indexed by dates):
# rets2 = add_stock_to_rets(rets, 'MY_STOCK', returns_series=candidate_ret)
# weights2 = recompute_weights_from_rets(rets2)
# print(weights2.round(3))

# 2) If you have price series `candidate_px`:
# rets2 = add_stock_to_rets(rets, 'MY_STOCK', prices=candidate_px)
# weights2 = recompute_weights_from_rets(rets2)
# print(weights2.round(3))
# ...existing code...

In [10]:
import yfinance as yf
import pandas as pd

univ = ['RR','CRML','OPEN','ONDS','RGTI','ASTS','LPTH','PATH','USAR','UPS','OBDC','SRFM']
px = yf.download(univ, start="2020-01-01", auto_adjust=False)['Adj Close']    # auto_adjust=True recommended
rets = px.pct_change(fill_method=None)                                                         # clearer than px/px.shift()-1
rets = rets.dropna(how='all')       # drop rows where every ticker is NaN
rets

[*********************100%***********************]  12 of 12 completed


Ticker,ASTS,CRML,LPTH,OBDC,ONDS,OPEN,PATH,RGTI,RR,SRFM,UPS,USAR
Date,,,,,,,,,,,,
2020-01-03,0.002026,NaN,-0.013699,-0.021397,NaN,NaN,NaN,NaN,NaN,NaN,-0.000599,NaN
2020-01-06,0.000000,NaN,-0.027778,-0.012658,NaN,NaN,NaN,NaN,NaN,NaN,-0.004455,NaN
2020-01-07,0.003033,NaN,-0.014286,0.007576,NaN,NaN,NaN,NaN,NaN,NaN,-0.001721,NaN
2020-01-08,0.000000,NaN,0.000000,-0.010410,NaN,NaN,NaN,NaN,NaN,NaN,0.005690,NaN
2020-01-09,0.000000,NaN,0.000000,0.005844,NaN,NaN,NaN,NaN,NaN,NaN,0.002314,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-30,0.004914,-0.043077,-0.026994,-0.005452,-0.019060,-0.028049,0.065287,0.004722,0.046341,0.009412,-0.011479,-0.019954
2025-10-01,0.160147,0.139871,-0.031526,-0.020360,-0.053109,0.011292,-0.029148,0.002014,0.109557,0.044289,0.010176,0.070971
2025-10-02,0.161925,0.142454,0.063802,0.017586,0.259918,-0.006203,-0.018476,0.185930,0.084034,0.055804,0.014103,0.233569


In [11]:
stats = {}
stats['ret'] = rets.mean()*252
stats['vol'] = rets.std()*np.sqrt(252)
stats['SR'] = rets.mean() / rets.std()*np.sqrt(252)
stats = pd.DataFrame(stats)
stats

,ret,vol,SR
Ticker,,,
ASTS,0.801534,1.003583,0.798672
CRML,0.795602,1.293348,0.615149
LPTH,0.760490,0.853320,0.891213
OBDC,0.089066,0.275645,0.323119
ONDS,0.719422,1.117805,0.643603
OPEN,0.547200,1.130801,0.483905
PATH,-0.153487,0.630697,-0.243360
RGTI,1.117079,1.296361,0.861703
RR,1.858617,1.782949,1.042440


In [12]:
weights = recompute_weights_from_rets(rets)
weights

,opt,eqvol,sr
Ticker,,,
ASTS,0.079783,0.061749,0.105185
CRML,0.038636,0.047914,0.062865
LPTH,0.112921,0.072622,0.138042
OBDC,0.139573,0.224819,0.154936
ONDS,0.010561,0.055439,0.076101
OPEN,0.054423,0.054802,0.056561
PATH,-0.227957,0.098257,-0.051000
RGTI,0.051210,0.047803,0.087856
RR,0.069885,0.034757,0.077278


In [13]:
combo_rets={}
combo_rets['opt'] = (rets*weights['opt']).sum(1)
combo_rets['eqvol'] = (rets*weights['eqvol']).sum(1)
combo_rets['sr'] = (rets*weights['sr']).sum(1)
combo_rets = pd.DataFrame(combo_rets)
combo_sr = combo_rets.mean() / combo_rets.std() * np.sqrt(252)
combo_sr

opt      1.593821
eqvol    1.033765
sr       1.424236
dtype: float64

In [14]:
constant = 5
scaled = (rets*(weights['opt']*constant)).sum(1)
scaled.mean()/scaled.std()*np.sqrt(252)


np.float64(1.5938205478210703)